In [1]:
import os
import io
import re
import pandas as pd
import json

In [2]:
work_fldr = r'D:\Users\Work\OneDrive - Transport Systems Catapult\Research\PhD\Models\tripchains_from_od'
#work_fldr = r'C:\Users\haris.ballis\OneDrive - Transport Systems Catapult\Research\PhD\Models\tripchains_from_od'
#work_fldr = r'C:\Users\Work\Desktop\local_optimisation\tripchains_from_od'

data_source = 'mnd'
model_name = 'greenwich_withinLondon'
trips_direction = 'allDirections'
distr_contr_enabled_switch = [True, False]

# Dimensions used for the distribution comparison. List should have same order as in main function (of full script)
#distr_dims = ['NumLegs', 'DistanceGroup']#, 'TimePeriodStart', 'TimePeriodEnd']

global_separator = '-'

In [3]:
def prepare_paths(run_results_fldr):

    paths = dict()
    #paths['zones_info'] = os.path.join(run_results_fldr, 'zones_info.csv')
    paths['od'] = os.path.join(run_results_fldr, 'od_filtered.csv')
    #paths['tripchains_description'] = os.path.join(run_results_fldr, 'tripchains_description.csv')
    paths['tripchains_info'] = os.path.join(run_results_fldr, 'tripchains_info.csv')
    paths['target_distribution'] = os.path.join(run_results_fldr, 'target_distribution.csv')
    paths['optimisation_results'] = os.path.join(run_results_fldr, 'results.json')

    return paths

In [4]:
def get_optimisation_results(infilepath):
    with open(infilepath) as f:
        data = json.load(f)

        results = data['Solution'][1]['Variable']

        df_data = list()
        for chain, val in results.items():
            cid = re.search('\[(\d+)\]', chain, re.DOTALL).group(1)
            times_used = val['Value']
            df_data.append([int(cid), times_used])

        df = pd.DataFrame(df_data, columns=['TripChainId', 'TimesUsed']).set_index('TripChainId').squeeze()
        return df

In [5]:
def convert_optimisation_output_to_trips(tcs_info, optim_output):

    # Append how many times the tripchain has been used by the optimiser 
    df = tcs_info.join(optim_output, how='right')
    
    # Create list with the OdPairsId which consist each of the used chains 
    tc_od_pairs = [x.split(global_separator) for x in df.OdPairsId.values]
    # Zip each tripchain's description with the times used 
    times_used_tc_od_pairs = list(zip(tc_od_pairs, df.TimesUsed))
    
    # create a df which contains the OdPairId and how many times it has been used
    data = list()
    for tc_od_pairs, times_used in times_used_tc_od_pairs:
        for od_pair in tc_od_pairs:
            data.append([int(od_pair), int(times_used)])

    df = pd.DataFrame(data, columns=['OdPairId', 'TimesUsed'])
    optimisation_trips = df.groupby('OdPairId').sum().squeeze()
    optimisation_trips.name = 'OptimisationTrips'
    return optimisation_trips


In [6]:
def get_optimisation_trips(paths):
    
    tcs_info = pd.read_csv(paths['tripchains_info'], index_col=['TripChainId'])
    optim_output = get_optimisation_results(paths['optimisation_results'])
    
    if optim_output is not None:
        optimisation_trips = convert_optimisation_output_to_trips(tcs_info, optim_output)
        return optimisation_trips, optim_output
    else:
        return None

In [7]:
def include_run_info(df, legs, trips_thresh, distr_constr_enabled):
    try:
        df = df.to_frame()
    except AttributeError:
        pass
    
    df['MaxLegs'] = int(legs)
    df['TripsThresh'] = int(trips_thresh)
    df['DistrConstr'] = distr_constr_enabled
    df.set_index(['MaxLegs', 'TripsThresh', 'DistrConstr'], append=True, inplace=True)
    df = df.squeeze()
    
    return df

In [8]:
def prepare_ods_comparison(optim_ods, init_ods):
    comp = init_ods.join(optim_ods)
    comp.index = comp.index.droplevel('OdPairId')
    comp.set_index(['OrigZoneId', 'DestZoneId', 'Purpose', 'TimePeriod'], append=True, inplace=True)
    comp.columns.name = 'TripsSource'
    comp = comp.stack()
    comp.name = 'Trips'
    return comp

In [9]:
def get_input_params_from_folder(infldr):
    fldrs = [name for name in os.listdir(infldr) if os.path.isdir(os.path.join(infldr, name))]
    inputs = []
    for fldr in fldrs:
        matches = re.findall('(\d+)legs-over(\d+)trips-distrConstr(\w+)', fldr)
        if matches:
            legs = int(matches[0][0])
            trips = int(matches[0][1])
            if matches[0][2] == 'True':
                distr_constr_enabled = True
            elif matches[0][2] == 'False':
                distr_constr_enabled = False
            else:
                raise ValueError('distrConstrEnabled must be either True or False')
                
            inputs.append([legs, trips, distr_constr_enabled])
        
    return inputs

In [10]:
def prepare_ditribution_comparison(fldrpaths):
    tcs_info = pd.read_csv(fldrpaths['tripchains_info'], index_col=['TripChainId'])
    optim_output = get_optimisation_results(fldrpaths['optimisation_results'])
    target_distr = pd.read_csv(fldrpaths['target_distribution'], index_col='DistrGroupId')

    df = tcs_info.join(optim_output, how='right')
    optim_distr = df.groupby('DistrGroupId')['TimesUsed'].sum()
    optim_distr = optim_distr.div(optim_distr.sum())
    optim_distr.name = 'Percentage'
    
    # We need to identify the distribution dimensions cols (should be all apart form the Percentage)
    idx_cols = list()
    for c in target_distr.columns:
        if c != 'Percentage':
            idx_cols.append(c)
    
    target_distr.set_index(idx_cols, append=True, inplace=True)
    target_distr = target_distr.squeeze()

    distr_comp = target_distr.to_frame().join(optim_distr, rsuffix='_')
    distr_comp.columns = ['Original', 'Modelled']
    distr_comp.columns.name = 'Source'
    distr_comp = distr_comp.stack()
    distr_comp.name = 'Percentage'

    return distr_comp

In [11]:
##### MAIN
generation_inputs_fldr = os.path.join(work_fldr, r'generation\inputs')

results_fldr = os.path.join(work_fldr, r'generation/outputs', '{}\{}\{}'.format(data_source, model_name, trips_direction))
inputs = get_input_params_from_folder(results_fldr)

concat_optim_od = list()
concat_optim_nchains = list()
concat_init_ods = list()
concat_distr_comps = list()

for legs, trips_thresh, distr_constr_enabled in inputs:

    run_tmplt = '{}legs-over{}trips-distrConstr{}'.format(legs, trips_thresh, distr_constr_enabled)
    run_results_fldr = os.path.join(results_fldr, run_tmplt)
    
    fldrpaths = prepare_paths(run_results_fldr)

    optim_od, optim_nchains = get_optimisation_trips(fldrpaths)

    if optim_od is None:
        print('No results for legs:{}-trips:{}-distrConstr:{}'.format(legs, trips_thresh, distr_constr_enabled))
        continue

    optim_od = include_run_info(optim_od, legs, trips_thresh, distr_constr_enabled)
    optim_nchains = include_run_info(optim_nchains, legs, trips_thresh, distr_constr_enabled)

    concat_optim_od.append(optim_od)
    concat_optim_nchains.append(optim_nchains)

    idx_cols_od = ['OrigZoneId', 'DestZoneId', 'Purpose', 'TimePeriod']
    init_od = pd.read_csv(fldrpaths['od'], index_col='OdPairId')
    init_od = include_run_info(init_od, legs, trips_thresh, distr_constr_enabled)
    concat_init_ods.append(init_od)

    distr_comp = prepare_ditribution_comparison(fldrpaths) 
    distr_comp = include_run_info(distr_comp, legs, trips_thresh, distr_constr_enabled)
    concat_distr_comps.append(distr_comp)

optim_nchains = pd.concat(concat_optim_nchains)
optim_ods = pd.concat(concat_optim_od)
init_ods = pd.concat(concat_init_ods)
distr_comps = pd.concat(concat_distr_comps)

#zones_info = pd.read_csv(fldrpaths['zones_info'], index_col=['ZoneId'])

ods_comparison = prepare_ods_comparison(optim_ods, init_ods)

#trips_comparison = trips_comparison.reorder_levels(['Legs', 'TripsThresh', 'DistrConstr', 'OdPairId'])
#trips_comparison.columns.name = 'TripsSource'
#trips_comparison = trips_comparison.stack()
#trips_comparison.name = 'Trips'
#trips_comparison.to_csv('tmp.csv', header=True)

In [14]:
optim_nchains.to_csv(r'./outputs/optimisation_results.csv', header=True, index=True)

In [66]:
distr_comps.to_csv(r'./outputs/distributions_comparison_latest.csv', header=True, index=True)

In [29]:
df = distr_comps.xs([4, 50], level=['MaxLegs', 'TripsThresh'])
df

DistrGroupId  Source        DistrConstr
0             Target        False          0.411524
              Optimisation  False          0.059799
1             Target        False          0.024193
              Optimisation  False          0.001935
2             Target        False          0.005067
3             Target        False          0.252718
              Optimisation  False          0.173160
4             Target        False          0.034573
              Optimisation  False          0.036369
5             Target        False          0.011606
              Optimisation  False          0.005182
6             Target        False          0.194769
              Optimisation  False          0.250856
7             Target        False          0.046016
              Optimisation  False          0.200349
8             Target        False          0.019534
              Optimisation  False          0.272351
0             Target        True           0.411524
              Optimisati

In [25]:
#ods_comparison.to_csv(r'./outputs/comparison_{}_{}_{}_latest.csv'.format(data_source, model_name, trips_direction),
#                      header=True)

In [31]:
ods_comparison

MaxLegs  TripsThresh  DistrConstr  OrigZoneId  DestZoneId  Purpose           TimePeriod  TripsSource      
3        100          False        E02000313   E02000314   HB_FromHome       IP          DailyTrips           191.0
                                                                                         OptimisationTrips    191.0
                                                                             OP          DailyTrips           175.0
                                                                                         OptimisationTrips    175.0
                                                           HB_ToHome         IP          DailyTrips           191.0
                                                                                         OptimisationTrips    191.0
                                                                             OP          DailyTrips           175.0
                                                                                 

In [39]:
df = distr_comps.copy()
df = df.xs(True, level='DistrConstr')
df = df.unstack('Source')
df['Diff_perc'] = df.Target.sub(df.Optimisation)

tolerance = 0.01
df['BelowTolerance'] = df.Diff_perc.abs() <= tolerance
df['BelowTolerance'].all()

True

In [33]:
df = ods_comparison.groupby(level=['MaxLegs', 'TripsThresh', 'DistrConstr', 'TripsSource', 'Purpose']).sum()

df = df.unstack('TripsSource')

df['Diff'] = df.DailyTrips - df.OptimisationTrips
df['Diff_perc'] = df.OptimisationTrips/df.DailyTrips - 1

df.style.format({
    'OptimisationTrips': '{:,.0f}'.format,
    'DailyTrips': '{:,.0f}'.format,
    'Diff': '{:,.0f}'.format,
    'Diff_perc': '{:.0%}'.format
})

In [28]:
df.unstack('TripsThresh')

TripsSource                          DailyTrips                     \
TripsThresh                                 25        50       100   
MaxLegs DistrConstr Purpose                                          
3       False       HB_FromHome         86984.0   68684.0  50123.0   
                    HB_ToHome           93962.0   75572.0  55502.0   
                    NHB_NonHomeBased   161683.0  126199.0  85800.0   
        True        HB_FromHome         86984.0   68684.0  50123.0   
                    HB_ToHome           93962.0   75572.0  55502.0   
                    NHB_NonHomeBased   161683.0  126199.0  85800.0   
4       False       HB_FromHome         86984.0   68684.0  50123.0   
                    HB_ToHome           93962.0   75572.0  55502.0   
                    NHB_NonHomeBased   161683.0  126199.0  85800.0   
        True        HB_FromHome         86984.0   68684.0  50123.0   
                    HB_ToHome           93962.0   75572.0  55502.0   
                    NHB_NonHomeBased   161683.0  126199.0  85800.0   
5       False       HB_FromHome             NaN   68684.0  50123.0   
                    HB_ToHome               NaN   75572.0  55502.0   
                    NHB_NonHomeBased        NaN  126199.0  85800.0   
        True        HB_FromHome             NaN   68684.0  50123.0   
                    HB_ToHome               NaN   75572.0  55502.0   
                    NHB_NonHomeBased        NaN  126199.0  85800.0   

TripsSource                          OptimisationTrips                    \
TripsThresh                                        25       50       100   
MaxLegs DistrConstr Purpose                                                
3       False       HB_FromHome                80663.0  62175.0  43879.0   
                    HB_ToHome                  80663.0  62175.0  43879.0   
                    NHB_NonHomeBased           54977.0  34724.0  18208.0   
        True        HB_FromHome                45812.0  30327.0  16686.0   
                    HB_ToHome                  45812.0  30327.0  16686.0   
                    NHB_NonHomeBased            5848.0   3756.0   1980.0   
4       False       HB_FromHome                80784.0  62526.0  43914.0   
                    HB_ToHome                  80784.0  62526.0  43914.0   
                    NHB_NonHomeBased           81528.0  49628.0  24106.0   
        True        HB_FromHome                47576.0  31493.0  17251.0   
                    HB_ToHome                  47576.0  31493.0  17251.0   
                    NHB_NonHomeBased           10897.0   6998.0   3756.0   
5       False       HB_FromHome                    NaN  62484.0  43914.0   
                    HB_ToHome                      NaN  62484.0  43914.0   
                    NHB_NonHomeBased               NaN  51326.0  24478.0   
        True        HB_FromHome                    NaN  31753.0  17396.0   
                    HB_ToHome                      NaN  31753.0  17396.0   
                    NHB_NonHomeBased               NaN   9038.0   4738.0   

TripsSource                               Diff                    Diff_perc  \
TripsThresh                                25        50       100       25    
MaxLegs DistrConstr Purpose                                                   
3       False       HB_FromHome         6321.0    6509.0   6244.0 -0.072669   
                    HB_ToHome          13299.0   13397.0  11623.0 -0.141536   
                    NHB_NonHomeBased  106706.0   91475.0  67592.0 -0.659970   
        True        HB_FromHome        41172.0   38357.0  33437.0 -0.473328   
                    HB_ToHome          48150.0   45245.0  38816.0 -0.512441   
                    NHB_NonHomeBased  155835.0  122443.0  83820.0 -0.963830   
4       False       HB_FromHome         6200.0    6158.0   6209.0 -0.071277   
                    HB_ToHome          13178.0   13046.0  11588.0 -0.140248   
                    NHB_NonHomeBased   80155.0   76571.0  61694

In [29]:
optim_nchains.groupby(level=[1,2,3]).sum()

MaxLegs  TripsThresh  DistrConstr
3        25           False          80663.0
                      True           45812.0
         50           False          62175.0
                      True           30327.0
         100          False          43879.0
                      True           16686.0
4        25           False          80784.0
                      True           47576.0
         50           False          62526.0
                      True           31493.0
         100          False          43914.0
                      True           17251.0
5        50           False          62484.0
                      True           31753.0
         100          False          43914.0
                      True           17396.0
Name: TimesUsed, dtype: float64

In [30]:
optim_trips.xs(True, level='DistrConstr')

NameError: name 'optim_trips' is not defined